# Operations Reasearch: Examples for Gurobipy

## Producing desks and tables

Consider the problem we introduced in Operations Research: Modeling and Application, we have

$$
\begin{split}
    \begin{array}{r}
        \max \\ \text{s.t.} \\ \\ \\ \\ \\ 
    \end{array} &
    \begin{array}{rcrcll}
        700x_1 & + & 900x_2 & & & \\ 
        3x_1 & + & 5x_2 & \leq & 3600\quad & \text{(wood)} \\		
        x_1 & + & 2x_2 & \leq & 1600\quad & \text{(labor)} \\		
        50x_1 & + & 20x_2 & \leq & 48000\quad & \text{(machine)} \\
        x_1 & & & \geq & 0
        \\
        & & x_2 & \geq & 0.
    \end{array}
\end{split}
$$

Let's construct the problem step by step

 We should import the pyomo environ and pyomo opt

In [1]:
#import libraries to work with pyomo
from pyomo.environ import *
from pyomo.opt import SolverFactory


Use pyomo to solve optimization model

In [7]:
m = ConcreteModel()
m.x1 = Var(within=NonNegativeReals, name='x1')
m.x2 = Var(within=NonNegativeReals, name='x2')

Set the objective function and add some constraints. It's necessary to set whether a problem is a maximization or minimization program. Also, remember to give all constraints, variables and the model distinct names.

In [8]:
m.obj = Objective(expr= 700*m.x1 + 900*m.x2, sense=maximize)
m.con1 = Constraint(expr= 3*m.x1 + 5 * m.x2 <= 3600, name='resource_wood')
m.con2 = Constraint(expr= m.x1 + 2 * m.x2 <= 1600, name='resource_labor')
m.con3 = Constraint(expr= 50 * m.x1 + 20 * m.x2 <= 48000, name='resource_machine')

Use **optimize** to run and solve the model. Finally, we can use **getattr** to extract all of the variables and use the method **obj** to get the objective value.

In [9]:
#eg1.optimize()
#use gurobi to solve the model
opt = SolverFactory('gurobi')
results = opt.solve(m)

In [ ]:
for v in m.component_objects(Var, active=True):
    varobject = getattr(m, str(v))
    for index in varobject:
        print("Variable", varobject[index].name, "=", varobject[index].value)
print("objective value =", m.obj())

# for var in eg1.getVars():
#     print(var.varName, '=', var.x)
# print("objective value =", eg1.objVal)
# x1 = 884.2105263157895
# x2 = 189.4736842105263
# objective value = 789473.6842105263


Variable x1 = 884.2105263157895
Variable x2 = 189.4736842105263
objective value = 789473.6842105263


## Decopling the Model and the Data 

Now let's try to decoupling the data from the model. The data part is as below:

In [12]:
products = range(2)  # 2 products    
resources = range(3)  # 3 resources

prices = [700, 900]    
resource_consumptions = [[3 , 5 ],
                         [1 , 2 ],
                         [50, 20]]
resource_limitations = [3600, 1600, 48000]

We can rewrite our model in a simpler way.

In [ ]:
m_decoupling = ConcreteModel()
m_decoupling.x = Var(products, within=NonNegativeReals, name='x')
m_decoupling.obj = Objective(expr= sum(prices[i] * m_decoupling.x[i] for i in products), 
                             sense=maximize)
m_decoupling.con = ConstraintList()
for j in resources:
    m_decoupling.con.add(sum(resource_consumptions[j][i] * m_decoupling.x[i] for i in products) <= resource_limitations[j])

opt_decoupling = SolverFactory('gurobi')
#using tee parameter to print the optimization process
results_decoupling = opt_decoupling.solve(m_decoupling, tee=True)

#print the results
for v in m_decoupling.component_objects(Var, active=True):
    varobject = getattr(m_decoupling, str(v))
    for index in varobject:
        print("Variable", varobject[index].name, "=", varobject[index].value)
print("objective value =", m_decoupling.obj())

Read LP format model from file /var/folders/kq/6xfsxyb97s31bxqdbd2dvct80000gq/T/tmp0gpf8z45.pyomo.lp
Reading time = 0.00 seconds
x1: 3 rows, 2 columns, 6 nonzeros
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.5.0 25F71)

CPU model: Apple M5
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros (Max)
Model fingerprint: 0xfc16a329
Model has 2 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 5e+01]
  Objective range  [7e+02, 9e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+03, 5e+04]

Presolve time: 0.00s
Presolved: 3 rows, 2 columns, 6 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    2.0000000e+32   3.593750e+30   2.000000e+02      0s
       3    7.8947368e+05   0.000000e+00   0.000000e+00      0s

Solved in 3 iterations and 0.00 seconds (0.00 work units)
Optimal objective  7.894736842e+05
Variable 